In [ ]:
# Importar el dataset
from google.colab import files
uploaded = files.upload()

In [ ]:
# convertir el dataset a pandas
import pandas as pd
df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

# Ver 5 primeras filas
df.head()


In [ ]:
# Entender el dataset
print("Shape:", df.shape)
print("\nInfo:")
df.info()

In [ ]:
#eliminar filas sin texto
df = df.dropna(subset=["Review Text"])
print("Shape después de limpiar:", df.shape)

In [ ]:
# creacion variable objetivo
def label_y(rating):
    if rating >= 4:
        return 1
    return 0

df["label"] = df["Rating"].apply(label_y)

df[["Rating", "label"]].head()

In [ ]:
# Visualización para entender los datos
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df, x="label")
plt.title("Distribución de Sentimientos")
plt.show()

In [ ]:
df = df[["Review Text", "label"]]
df.head()

In [ ]:
!pip install transformers torch accelerate -q

In [ ]:
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import pipeline

In [ ]:
# division de datos en entrenamiento y prueba
X = df["Review Text"].astype(str)
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
X_eval = X_test[:500].tolist()
y_eval = y_test[:500].tolist()

In [ ]:
def evaluar_modelo(nombre_modelo, modelo_hf, tipo_salida="positive_negative"):

    print(f"Evaluando: {nombre_modelo}")

    clasificador = pipeline(
        "sentiment-analysis",
        model=modelo_hf,
        truncation=True
    )

    predicciones = []
    inicio = time.time()

    for texto in X_eval:
        resultado = clasificador(texto)[0]
        etiqueta = resultado["label"]

        if tipo_salida == "positive_negative":
            if etiqueta.upper() == "POSITIVE":
                predicciones.append(1)
            else:
                predicciones.append(0)

        elif tipo_salida == "stars":
            estrellas = int(etiqueta[0])

            if estrellas >= 4:
                predicciones.append(1)
            else:
                predicciones.append(0)

    fin = time.time()

    return {
        "Modelo": nombre_modelo,
        "Accuracy": round(accuracy_score(y_eval, predicciones), 4),
        "Precision": round(precision_score(y_eval, predicciones), 4),
        "Recall": round(recall_score(y_eval, predicciones), 4),
        "F1-score": round(f1_score(y_eval, predicciones), 4),
        "Tiempo_segundos": round(fin - inicio, 2)
    }

In [ ]:
resultados = []

resultados.append(
    evaluar_modelo(
        "DistilBERT SST-2",
        "distilbert-base-uncased-finetuned-sst-2-english",
        "positive_negative"
    )
)

resultados.append(
    evaluar_modelo(
        "RoBERTa Twitter Sentiment",
        "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "positive_negative"
    )
)

resultados.append(
    evaluar_modelo(
        "BERT Multilingual Stars",
        "nlptown/bert-base-multilingual-uncased-sentiment",
        "stars"
    )
)

In [ ]:
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados = tabla_resultados.sort_values(
    by="F1-score",
    ascending=False
)

tabla_resultados

In [ ]:
mejor_modelo = tabla_resultados.iloc[0]
mejor_modelo

In [ ]:
nombre_modelo_ganador = tabla_resultados.iloc[0]["Modelo"]

print(nombre_modelo_ganador)

In [ ]:
modelo_final = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

In [ ]:
nuevas_resenas = [
    "This dress is amazing and fits perfectly",
    "Terrible quality and very uncomfortable"
]

for texto in nuevas_resenas:
    resultado = modelo_final(texto)[0]
    estrellas = int(resultado["label"][0])

    sentimiento = 1 if estrellas >= 4 else 0
    etiqueta = "Positivo" if sentimiento == 1 else "Negativo"

    print("Texto:", texto)
    print("Predicción:", etiqueta)
    print("Estrellas:", resultado["label"])
    print("Confianza:", round(resultado["score"], 4))
    print()